# Tema: MERGE de clientes

## Objetivos
Actualizar e insertar por clave, resolver duplicados y bloquear eventos antiguos.

## Conceptos importantes para el examen
WHEN MATCHED; WHEN NOT MATCHED; condiciones de negocio; fuente única por clave.

**Dificultad:** Intermedio · **Tiempo estimado:** 75 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_06_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
customers = spark.createDataFrame(
    [(i, f"Cliente {i:02d}", ["Madrid", "Sevilla", "Bilbao"][i % 3],
      datetime(2026, 1, 1)) for i in range(1, 13)],
    "customer_id INT, name STRING, city STRING, updated_at TIMESTAMP"
)
customers.write.format("delta").mode("errorifexists").saveAsTable("customers")
updates = spark.createDataFrame([
    (2, "Cliente 02", "Valencia", datetime(2026, 2, 1)),
    (5, "Cliente 05", "Zaragoza", datetime(2026, 2, 2)),
    (13, "Cliente 13", "Madrid", datetime(2026, 2, 3))],
    customers.schema)
updates.write.format("delta").mode("errorifexists").saveAsTable("customers_updates")
display(customers)

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Upsert básico

In [ ]:
%sql
CREATE TABLE customers_demo USING DELTA AS SELECT * FROM customers;
MERGE INTO customers_demo t USING customers_updates s ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET t.name = s.name, t.city = s.city, t.updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (customer_id, name, city, updated_at) VALUES (s.customer_id, s.name, s.city, s.updated_at);

### 2. Evitar sobrescribir con eventos antiguos

In [ ]:
%sql
MERGE INTO customers_demo t USING customers_updates s ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
SELECT * FROM customers_demo ORDER BY customer_id;

### 3. Reducir una fuente con duplicados

In [ ]:
late = updates.filter("customer_id = 2").withColumn("city", F.lit("Granada")).withColumn("updated_at", F.to_timestamp(F.lit("2026-03-01")))
duplicated = updates.unionByName(late)
w = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
latest_updates = duplicated.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
latest_updates.createOrReplaceTempView("latest_updates")
display(latest_updates)

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Crea customers_practice desde customers y aplica el upsert de customers_updates. Deben quedar 13 clientes.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Aplica latest_updates solo cuando sea más reciente. El cliente 2 debe terminar en Granada.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Repite el MERGE anterior y verifica que los datos son iguales antes y después.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Simula un cambio antiguo del cliente 2 con fecha 2025-12-01. Demuestra que no sobrescribe Granada.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Añade un evento DELETE para el cliente 5 y uno UPSERT para el 14; aplica ambos en un MERGE.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** ON compara la clave.

**Pista 2:** Condición adicional en WHEN MATCHED.

**Pista 3:** exceptAll en ambas direcciones; materializa el antes en otra tabla.

**Pista 4:** Conserva el predicado temporal.

**Pista 5:** La rama DELETE precede a la actualización general.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
%sql
CREATE OR REPLACE TABLE customers_practice USING DELTA AS SELECT * FROM customers;
MERGE INTO customers_practice t USING customers_updates s ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
SELECT COUNT(*) AS n FROM customers_practice;

### Solución 2

In [ ]:
%sql
MERGE INTO customers_practice t USING latest_updates s ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

### Solución 3

In [ ]:
spark.sql("CREATE OR REPLACE TABLE before_replay USING DELTA AS SELECT * FROM customers_practice")
spark.sql("""MERGE INTO customers_practice t USING latest_updates s ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
a, b = spark.table("before_replay"), spark.table("customers_practice")
assert a.exceptAll(b).count() == b.exceptAll(a).count() == 0

### Solución 4

In [ ]:
updates.filter("customer_id = 2").withColumn("updated_at", F.to_timestamp(F.lit("2025-12-01"))).createOrReplaceTempView("old_updates")
spark.sql("""MERGE INTO customers_practice t USING old_updates s ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
assert spark.table("customers_practice").filter("customer_id = 2").first().city == "Granada"

### Solución 5

In [ ]:
spark.sql("""CREATE OR REPLACE TEMP VIEW changes AS
SELECT 5 customer_id, 'Cliente 05' name, 'Zaragoza' city, TIMESTAMP '2026-04-01' updated_at, 'DELETE' op
UNION ALL SELECT 14, 'Cliente 14', 'Cádiz', TIMESTAMP '2026-04-01', 'UPSERT'""")
spark.sql("""MERGE INTO customers_practice t USING changes s ON t.customer_id = s.customer_id
WHEN MATCHED AND s.op = 'DELETE' AND s.updated_at > t.updated_at THEN DELETE
WHEN MATCHED AND s.op = 'UPSERT' AND s.updated_at > t.updated_at THEN UPDATE SET t.name=s.name, t.city=s.city, t.updated_at=s.updated_at
WHEN NOT MATCHED AND s.op = 'UPSERT' THEN INSERT (customer_id,name,city,updated_at) VALUES(s.customer_id,s.name,s.city,s.updated_at)""")
assert spark.table("customers_practice").filter("customer_id = 5").count() == 0

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
Dos cambios de una fuente coinciden con el mismo destino. ¿Qué debes hacer?

A. Aumentar RAM siempre

B. Elegir una versión por clave antes del MERGE

C. Eliminar ON

D. Usar INSERT sin condición

### Pregunta 2
¿Dónde condicionas una actualización a un evento más reciente?

A. LOCATION

B. USING DELTA

C. WHEN MATCHED AND

D. DESCRIBE

### Pregunta 3
¿Qué rama inserta una nueva clave?

A. WHEN NOT MATCHED

B. WHEN MATCHED

C. ORDER BY

D. WHEN NOT MATCHED BY SOURCE siempre

### Respuestas y explicación
**1. B** — La fuente debe permitir una actualización no ambigua.

**2. C** — La condición protege el estado existente.

**3. A** — No existe una fila destino para esa clave.

## PARTE 6 - RETO FINAL
Recibe dos lotes desordenados con duplicados, altas y bajas. Diseña un upsert reproducible y explica qué estado adicional necesitarías para impedir que un evento antiguo resucite una baja.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
